In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder

In [2]:
# Set visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette('deep')

In [3]:
# Load the CSV file
df = pd.read_csv('survey_data.csv')
df.head()

,Schedule ID,FSU Serial No.,Schedule,survey year,Sector,NSS-Region,District,Stratum,Sub-Stratum,Sub-Round,...,Primary source of energey for lighting,Type of washing of clothes,Type of sweeping of floor,Dwelling unit,Type of structure of the dwelling unit,Is there any member in the household aged 5 years and above who needs special care,Is there any care giver available among the household members for caring the person(s),Time to canvass(minutes),NSC,MULT
0,TUS,32223,106,2024,1,11,21,13,1,2,...,1.0,2.0,2.0,1.0,2.0,2.0,NaN,90,4,67243
1,TUS,32223,106,2024,1,11,21,13,1,2,...,1.0,2.0,2.0,1.0,1.0,2.0,NaN,35,4,67243
2,TUS,32223,106,2024,1,11,21,13,1,2,...,1.0,1.0,2.0,1.0,3.0,2.0,NaN,90,4,67243
3,TUS,32223,106,2024,1,11,21,13,1,2,...,1.0,2.0,2.0,1.0,3.0,2.0,NaN,90,4,67243
4,TUS,32223,106,2024,1,11,21,13,1,2,...,1.0,1.0,2.0,1.0,3.0,2.0,NaN,90,4,67243


In [4]:
# Check data types and missing values
print("\nData Info:")
print(df.info())


Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139489 entries, 0 to 139488
Data columns (total 37 columns):
 #   Column                                                                                              Non-Null Count   Dtype  
---  ------                                                                                              --------------   -----  
 0   Schedule ID                                                                                         139489 non-null  object 
 1   FSU Serial No.                                                                                      139489 non-null  int64  
 2   Schedule                                                                                            139489 non-null  int64  
 3   survey year                                                                                         139489 non-null  int64  
 4   Sector                                                                                      

In [5]:
# Summary statistics
print("\nSummary Statistics:")
print(df.describe())


Summary Statistics:
       FSU Serial No.  Schedule  survey year         Sector     NSS-Region  \
count   139489.000000  139489.0     139489.0  139489.000000  139489.000000   
mean     46374.230864     106.0       2024.0       1.403200     192.465972   
std      14703.290452       0.0          0.0       0.490542      94.077912   
min      30010.000000     106.0       2024.0       1.000000      11.000000   
25%      33650.000000     106.0       2024.0       1.000000      95.000000   
50%      36930.000000     106.0       2024.0       1.000000     195.000000   
75%      63186.000000     106.0       2024.0       2.000000     274.000000   
max      66999.000000     106.0       2024.0       2.000000     371.000000   

            District        Stratum    Sub-Stratum      Sub-Round  \
count  139489.000000  139489.000000  139489.000000  139489.000000   
mean       16.730559      16.671121       9.764942       2.501065   
std        13.747657      15.758972      13.128614       1.118169   


Data cleaning

In [6]:
# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
Schedule ID                                                                                                0
FSU Serial No.                                                                                             0
Schedule                                                                                                   0
survey year                                                                                                0
Sector                                                                                                     0
NSS-Region                                                                                                 0
District                                                                                                   0
Stratum                                                                                                    0
Sub-Stratum                                                                                                0
Su

In [7]:
# Handle missing values in expenditure columns (fill with 0)
expenditure_cols = [
    'usual consumer expenditure in a month for household purposes out of purchase (A)',
    'imputed value of usual consumption in a month from home grown stock (B)',
    'imputed value of usual consumption in a month from wages in kind, free collection, gifts, etc (C )',
    'expenditure on purchase of items like clothing, footwear etc. during last 365 days (D)',
    'expenditure on purchase of household durable during last 365 days (E)',
    'usual monthly consumer expenditure E: [A+B+C+(D+E)/12]'
]
for col in expenditure_cols:
    df[col] = df[col].fillna(0).astype(float)

In [8]:
df.head()

,Schedule ID,FSU Serial No.,Schedule,survey year,Sector,NSS-Region,District,Stratum,Sub-Stratum,Sub-Round,...,Primary source of energey for lighting,Type of washing of clothes,Type of sweeping of floor,Dwelling unit,Type of structure of the dwelling unit,Is there any member in the household aged 5 years and above who needs special care,Is there any care giver available among the household members for caring the person(s),Time to canvass(minutes),NSC,MULT
0,TUS,32223,106,2024,1,11,21,13,1,2,...,1.0,2.0,2.0,1.0,2.0,2.0,NaN,90,4,67243
1,TUS,32223,106,2024,1,11,21,13,1,2,...,1.0,2.0,2.0,1.0,1.0,2.0,NaN,35,4,67243
2,TUS,32223,106,2024,1,11,21,13,1,2,...,1.0,1.0,2.0,1.0,3.0,2.0,NaN,90,4,67243
3,TUS,32223,106,2024,1,11,21,13,1,2,...,1.0,2.0,2.0,1.0,3.0,2.0,NaN,90,4,67243
4,TUS,32223,106,2024,1,11,21,13,1,2,...,1.0,1.0,2.0,1.0,3.0,2.0,NaN,90,4,67243


In [9]:
# Strip leading/trailing spaces from column names
df.columns = df.columns.str.strip()

# Handle missing values in categorical columns (fill with mode or 'Unknown')
categorical_cols = [
    'religion', 'Social group', 'Land possessed as on date of survey(code)',
    'Primary source of energey for cooking', 'Primary source of energey for lighting',
    'Type of washing of clothes', 'Type of sweeping of floor', 'Dwelling unit',
    'Type of structure of the dwelling unit'
]
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0]).astype('category')


In [10]:
# Fill missing values in 'Household size' with mode, then convert to integer
df['Household size'] = df['Household size'].fillna(df['Household size'].mode()[0]).astype(int)

In [11]:
# Drop irrelevant columns (e.g., IDs, codes not needed for analysis)
drop_cols = ['Schedule ID', 'FSU Serial No.', 'Sample hhld. No.', 'Informant Sl.No.', 'NSC', 'MULT']
df = df.drop(columns=drop_cols)

In [12]:
# Check cleaned data
print("\nCleaned Data Info:")
print(df.info())


Cleaned Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139489 entries, 0 to 139488
Data columns (total 31 columns):
 #   Column                                                                                              Non-Null Count   Dtype   
---  ------                                                                                              --------------   -----   
 0   Schedule                                                                                            139489 non-null  int64   
 1   survey year                                                                                         139489 non-null  int64   
 2   Sector                                                                                              139489 non-null  int64   
 3   NSS-Region                                                                                          139489 non-null  int64   
 4   District                                                                      

In [13]:
import json
from collections import Counter

# Load the JSON file
with open('daikibo_telemetry.json') as f:
    data = json.load(f)

# Quick check of a sample
print(data[0])


FileNotFoundError: [Errno 2] No such file or directory: 'daikibo_telemetry.json'